# 🎯 PSC AI-Agent — Fine-Tuning Notebook
**مدل:** Llama 3.1 8B Instruct (4-bit) via Unsloth  
**داده:** PSC_AI_MASTER_v4_optimized.jsonl (275 entries)  
**ذخیره:** Google Drive + Hugging Face Hub  
**سخت‌افزار:** T4 GPU (Free) یا P100/A100 (Pro)

---
### 📋 مراحل:
1. نصب کتابخانه‌ها
2. اتصال به Google Drive
3. بارگذاری داده PSC
4. بارگذاری و پیکربندی مدل
5. Fine-tuning با LoRA
6. ارزیابی و تست
7. ذخیره در Drive و Hugging Face

## 🔧 مرحله ۱ — بررسی GPU و نصب Unsloth

In [ ]:
# بررسی GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout[:500] if result.returncode == 0 else "No GPU detected")

In [ ]:
# نصب Unsloth (نسخه بهینه برای Colab T4)
%%capture
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install",
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git",
    "--quiet"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install",
    "--no-deps", "trl", "peft", "accelerate", "bitsandbytes",
    "--quiet"], check=True)

print("✅ Unsloth installed")

## 📂 مرحله ۲ — اتصال Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── مسیر پروژه در Drive ────────────────────────────────────────
DRIVE_BASE    = "/content/drive/MyDrive/PSC_AI_Agent"
DATA_DIR      = f"{DRIVE_BASE}/training_data"
OUTPUT_DIR    = f"{DRIVE_BASE}/models"
LOG_DIR       = f"{DRIVE_BASE}/logs"
CHECKPOINT_DIR= f"{DRIVE_BASE}/checkpoints"

for d in [DATA_DIR, OUTPUT_DIR, LOG_DIR, CHECKPOINT_DIR]:
    os.makedirs(d, exist_ok=True)

print("✅ Google Drive mounted")
print(f"📁 Project path: {DRIVE_BASE}")
print(f"📁 Data:   {DATA_DIR}")
print(f"📁 Output: {OUTPUT_DIR}")

## 📊 مرحله ۳ — بارگذاری داده PSC

In [ ]:
import json, shutil

# ── آپلود فایل JSONL از سیستم محلی یا Drive ──────────────────
JSONL_FILENAME = "PSC_AI_MASTER_v4_optimized.jsonl"
LOCAL_PATH  = f"/content/{JSONL_FILENAME}"
DRIVE_PATH  = f"{DATA_DIR}/{JSONL_FILENAME}"

# اگر فایل در Drive موجود است، از آنجا استفاده کن
if os.path.exists(DRIVE_PATH):
    shutil.copy(DRIVE_PATH, LOCAL_PATH)
    print(f"✅ Loaded from Drive: {DRIVE_PATH}")
else:
    # آپلود دستی از کامپیوتر محلی
    from google.colab import files
    print("⬆️  Upload PSC_AI_MASTER_v4_optimized.jsonl now:")
    uploaded = files.upload()
    for fname in uploaded:
        shutil.copy(fname, DRIVE_PATH)
        shutil.copy(fname, LOCAL_PATH)
    print(f"✅ Saved to Drive: {DRIVE_PATH}")

# بارگذاری و نمایش آمار
entries = []
with open(LOCAL_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            try: entries.append(json.loads(line))
            except: pass

print(f"\n📊 Dataset stats:")
print(f"  Total entries : {len(entries)}")
print(f"  Keys          : {list(entries[0].keys())}")
print(f"  Sample system : {entries[0]['messages'][0]['content'][:80]}...")
print(f"  Sample user   : {entries[0]['messages'][1]['content'][:100]}...")

In [ ]:
from datasets import Dataset

def format_entry(entry):
    """تبدیل هر entry به فرمت chat string برای Unsloth"""
    msgs = entry.get("messages", [])
    parts = []
    for m in msgs:
        role = m.get("role","")
        content = m.get("content","")
        if role == "system":
            parts.append(f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n{content}<|eot_id|>")
        elif role == "user":
            parts.append(f"<|start_header_id|>user<|end_header_id|>\n{content}<|eot_id|>")
        elif role == "assistant":
            parts.append(f"<|start_header_id|>assistant<|end_header_id|>\n{content}<|eot_id|>")
    return {"text": "\n".join(parts)}

formatted = [format_entry(e) for e in entries]
dataset = Dataset.from_list(formatted)

# Split 90/10 train/eval
split = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = split["train"]
eval_ds  = split["test"]

print(f"\n✅ Dataset ready:")
print(f"  Train : {len(train_ds)} entries")
print(f"  Eval  : {len(eval_ds)} entries")
print(f"  Sample:\n{train_ds[0]['text'][:300]}...")

## 🤖 مرحله ۴ — بارگذاری مدل Llama 3.1 8B

In [ ]:
from unsloth import FastLanguageModel
import torch

# ── پیکربندی مدل ──────────────────────────────────────────────
MODEL_NAME  = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
MAX_SEQ_LEN = 2048        # برای T4 مناسب (با P100/A100 می‌توان ۴۰۹۶ گذاشت)
LOAD_4BIT   = True        # ضروری برای T4 (12GB VRAM)

print(f"Loading {MODEL_NAME} ...")
print(f"Max seq length: {MAX_SEQ_LEN} | 4-bit: {LOAD_4BIT}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = None,        # خودکار (float16 روی T4)
    load_in_4bit    = LOAD_4BIT,
)

print(f"\n✅ Model loaded")
print(f"   Parameters: {model.num_parameters():,}")
print(f"   Device    : {next(model.parameters()).device}")

In [ ]:
# ── اضافه کردن لایه‌های LoRA ──────────────────────────────────
# r=16 برای T4 مناسب | r=32 برای P100/A100
LORA_R = 16

model = FastLanguageModel.get_peft_model(
    model,
    r                   = LORA_R,
    target_modules      = ["q_proj", "k_proj", "v_proj", "o_proj",
                           "gate_proj", "up_proj", "down_proj"],
    lora_alpha          = LORA_R * 2,     # معمولاً 2x rank
    lora_dropout        = 0.05,
    bias                = "none",
    use_gradient_checkpointing = "unsloth",  # صرفه‌جویی در VRAM
    random_state        = 42,
    use_rslora          = True,           # Rank-Stabilized LoRA
    loftq_config        = None,
)

# نمایش پارامترهای قابل آموزش
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"\n✅ LoRA configured (r={LORA_R})")
print(f"   Trainable params : {trainable:,} ({100*trainable/total:.2f}%)")
print(f"   Total params     : {total:,}")
print(f"   Frozen params    : {total-trainable:,}")

## 🏋️ مرحله ۵ — Fine-Tuning

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

# ── تشخیص خودکار قابلیت‌های GPU ──────────────────────────────
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
is_a100  = "A100" in gpu_name
is_p100  = "P100" in gpu_name or "V100" in gpu_name
print(f"GPU: {gpu_name}")

# ── تنظیمات بهینه برای هر GPU ────────────────────────────────
if is_a100:
    batch, grad_acc, epochs = 4, 2, 3
    fp16, bf16 = False, True
    print("Config: A100 — high performance")
elif is_p100:
    batch, grad_acc, epochs = 2, 4, 3
    fp16, bf16 = True, False
    print("Config: P100 — medium performance")
else:  # T4 (Free tier)
    batch, grad_acc, epochs = 1, 8, 2
    fp16, bf16 = True, False
    print("Config: T4 — memory-efficient")

print(f"Effective batch size: {batch * grad_acc}")
print(f"Epochs: {epochs}")

In [ ]:
import os
from datetime import datetime

RUN_NAME   = f"psc-llama31-8b-{datetime.now().strftime('%Y%m%d-%H%M')}"
CKPT_DIR   = f"{CHECKPOINT_DIR}/{RUN_NAME}"
os.makedirs(CKPT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir                  = CKPT_DIR,
    run_name                    = RUN_NAME,
    num_train_epochs            = epochs,
    per_device_train_batch_size = batch,
    per_device_eval_batch_size  = 1,
    gradient_accumulation_steps = grad_acc,
    warmup_ratio                = 0.05,
    learning_rate               = 2e-4,
    lr_scheduler_type           = "cosine",
    fp16                        = fp16,
    bf16                        = bf16,
    logging_steps               = 10,
    evaluation_strategy         = "steps",
    eval_steps                  = 50,
    save_strategy               = "steps",
    save_steps                  = 50,
    save_total_limit            = 3,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    optim                       = "adamw_8bit",
    weight_decay                = 0.01,
    max_grad_norm               = 1.0,
    seed                        = 42,
    report_to                   = "none",         # TensorBoard اگر خواستید "tensorboard"
    dataloader_num_workers      = 0,
)

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = train_ds,
    eval_dataset    = eval_ds,
    dataset_text_field = "text",
    max_seq_length  = MAX_SEQ_LEN,
    packing         = True,          # بهره‌وری بیشتر روی T4
    args            = training_args,
    data_collator   = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)

print(f"✅ Trainer ready | Run: {RUN_NAME}")
print(f"   Steps per epoch  : {len(trainer.get_train_dataloader())}")
print(f"   Total steps      : {trainer.args.max_steps if trainer.args.max_steps>0 else 'auto'}")

In [ ]:
# ── شروع آموزش ────────────────────────────────────────────────
print("🚀 Starting fine-tuning...")
print("="*50)

trainer_stats = trainer.train()

print("\n" + "="*50)
print("✅ Training complete!")
print(f"   Runtime   : {trainer_stats.metrics['train_runtime']:.0f}s ({trainer_stats.metrics['train_runtime']/3600:.2f}h)")
print(f"   Samples/s : {trainer_stats.metrics['train_samples_per_second']:.2f}")
print(f"   Final loss: {trainer_stats.metrics['train_loss']:.4f}")

## 📊 مرحله ۶ — ارزیابی و تست

In [ ]:
# ── ارزیابی نهایی ─────────────────────────────────────────────
eval_results = trainer.evaluate()
print(f"\n📊 Evaluation results:")
for k, v in eval_results.items():
    print(f"   {k}: {v:.4f}" if isinstance(v, float) else f"   {k}: {v}")

In [ ]:
# ── تست تولید پاسخ ────────────────────────────────────────────
FastLanguageModel.for_inference(model)

def test_psc(user_input, system_msg=None, max_new_tokens=512):
    if system_msg is None:
        system_msg = "تو PSC AI-Agent هستی — یک دستیار روان‌شناختی مبتنی بر مدل PSC V12. پاسخ دقیق و کاربردی بده."
    
    prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n{system_msg}<|eot_id|>\n<|start_header_id|>user<|end_header_id|>\n{user_input}<|eot_id|>\n<|start_header_id|>assistant<|end_header_id|>\n"
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens  = max_new_tokens,
            temperature     = 0.7,
            top_p           = 0.9,
            repetition_penalty = 1.1,
            do_sample       = True,
            pad_token_id    = tokenizer.eos_token_id,
        )
    
    full = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # استخراج پاسخ بعد از assistant header
    reply_start = full.find("\nassistant\n")
    if reply_start != -1:
        return full[reply_start+len("\nassistant\n"):]
    return full[len(prompt):]

# ── تست ۱: سطح ۱ PSC
print("TEST 1 — Level 1 PSC")
print("-"*50)
reply1 = test_psc("من دیابت دارم ولی وقتی استرس می‌گیرم شیرینی می‌خورم و بعد پشیمان می‌شوم. چه کار کنم؟")
print(reply1[:600])
print()

# ── تست ۲: محور HPA
print("TEST 2 — HPA Axis")
print("-"*50)
reply2 = test_psc("ارتباط کورتیزول با سطح ۱ PSC را توضیح بده")
print(reply2[:400])

## 💾 مرحله ۷ — ذخیره مدل

In [ ]:
# ── ذخیره LoRA adapters در Drive ─────────────────────────────
SAVE_NAME   = f"psc-llama31-8b-lora-{datetime.now().strftime('%Y%m%d')}"
LORA_PATH   = f"{OUTPUT_DIR}/{SAVE_NAME}"
GGUF_PATH   = f"{OUTPUT_DIR}/{SAVE_NAME}-q4"

model.save_pretrained(LORA_PATH)
tokenizer.save_pretrained(LORA_PATH)

print(f"✅ LoRA adapters saved: {LORA_PATH}")
print(f"   Size: {sum(os.path.getsize(os.path.join(dp,f)) for dp,dn,fn in os.walk(LORA_PATH) for f in fn)/1024/1024:.1f} MB")

In [ ]:
# ── ذخیره به فرمت GGUF (برای اجرا روی سرور معمولی بدون GPU) ─
print("Converting to GGUF q4_k_m (برای Render/Koyeb)...")

try:
    model.save_pretrained_gguf(
        GGUF_PATH,
        tokenizer,
        quantization_method = "q4_k_m",   # حجم کم + کیفیت مناسب
    )
    gguf_files = [f for f in os.listdir(GGUF_PATH) if f.endswith('.gguf')]
    total_mb = sum(os.path.getsize(f"{GGUF_PATH}/{f}") for f in gguf_files) / 1024/1024
    print(f"✅ GGUF saved: {GGUF_PATH}")
    print(f"   Files: {gguf_files}")
    print(f"   Size : {total_mb:.0f} MB")
except Exception as e:
    print(f"⚠️  GGUF conversion failed (optional): {e}")
    print("   LoRA adapters are still saved in Drive ✅")

In [ ]:
# ── آپلود به Hugging Face Hub (اختیاری) ──────────────────────
UPLOAD_TO_HF = False  # True کنید اگر می‌خواهید آپلود شود

if UPLOAD_TO_HF:
    from huggingface_hub import login
    
    # کلید HF خود را اینجا بگذارید
    HF_TOKEN    = ""   # از https://huggingface.co/settings/tokens بگیرید
    HF_USERNAME = ""   # نام کاربری Hugging Face شما
    MODEL_REPO  = f"{HF_USERNAME}/psc-ai-agent-llama31-8b"
    
    login(token=HF_TOKEN)
    
    model.push_to_hub(MODEL_REPO, tokenizer=tokenizer, token=HF_TOKEN)
    tokenizer.push_to_hub(MODEL_REPO, token=HF_TOKEN)
    
    print(f"✅ Uploaded to HF: https://huggingface.co/{MODEL_REPO}")
else:
    print("ℹ️  HF upload skipped (UPLOAD_TO_HF=False)")
    print(f"   Model available at: {LORA_PATH}")

## 🔄 مرحله ۸ — راه‌اندازی مجدد (Resume Training)

In [ ]:
# ── ادامه آموزش از آخرین checkpoint ──────────────────────────
# (اگر Colab قطع شد، این cell را اجرا کنید)

RESUME_TRAINING = False   # True کنید برای ادامه از checkpoint

if RESUME_TRAINING:
    # پیدا کردن آخرین checkpoint
    import glob
    ckpts = sorted(glob.glob(f"{CHECKPOINT_DIR}/*/checkpoint-*"))
    if ckpts:
        latest = ckpts[-1]
        print(f"Resuming from: {latest}")
        trainer_stats = trainer.train(resume_from_checkpoint=latest)
        print("✅ Training resumed and completed")
    else:
        print("❌ No checkpoints found")
else:
    print("ℹ️  Resume skipped")

## ✅ خلاصه نهایی

پس از اتمام موفق، فایل‌های زیر در Google Drive ذخیره شده‌اند:

```
MyDrive/PSC_AI_Agent/
├── training_data/
│   └── PSC_AI_MASTER_v4_optimized.jsonl
├── models/
│   ├── psc-llama31-8b-lora-YYYYMMDD/   ← LoRA adapters (~80MB)
│   └── psc-llama31-8b-lora-YYYYMMDD-q4/ ← GGUF q4_k_m (~4.5GB)
├── checkpoints/
│   └── psc-llama31-8b-YYYYMMDD-HHMM/
└── logs/
```

### استفاده از مدل آموزش‌دیده در Backend

مدل GGUF را می‌توانید با `llama-cpp-python` در FastAPI backend اجرا کنید:
```python
from llama_cpp import Llama
llm = Llama(model_path="psc-llama31-8b-lora-q4_k_m.gguf", n_ctx=2048)
```
